# Chest X-Ray Bone Suppression: Knowledge Distillation Pipeline
Pretrained ResNet34-UNet student distillation with bilinear interpolation decoder and bone-weighted gradient loss.

## 1. Setup & Dependencies

In [ ]:
!nvidia-smi
!pip install -q huggingface_hub torchvision opencv-python-headless matplotlib datasets tqdm

## 2. Load Teacher Model

In [ ]:
import os
import torch
from huggingface_hub import hf_hub_download

teacher_path = hf_hub_download(
    repo_id="qureaiorg/bone-suppression",
    filename="weights/bone_suppression.ts"
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
teacher_model = torch.jit.load(teacher_path, map_location=device).eval()
print(f"[INFO] Teacher model loaded on {device}")

## 3. Dataset Ingestion & Caching

In [ ]:
import cv2
import numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm

NUM_SAMPLES = 600
dataset_raw = load_dataset("hf-vision/chest-xray-pneumonia", split=f"train[:{NUM_SAMPLES}]")

raw_images = []
for sample in tqdm(dataset_raw, desc="Preprocessing CXR (1024x1024)"):
    img = sample['image'].convert('L')
    arr = np.array(img, dtype=np.float32)
    arr_resized = cv2.resize(arr, (1024, 1024), interpolation=cv2.INTER_AREA)
    lo, hi = float(arr_resized.min()), float(arr_resized.max())
    norm = np.clip((arr_resized - lo) / max(hi - lo, 1e-6) * 255.0, 0.0, 255.0).astype(np.uint8)
    raw_images.append(norm)

print(f"[INFO] Cached {len(raw_images)} images in memory (uint8 format)")

## 4. Precompute Supervision Targets

In [ ]:
cached_targets = []

with torch.no_grad():
    for img_u8 in tqdm(raw_images, desc="Extracting Bone Targets"):
        x_f32 = img_u8.astype(np.float32) / 255.0
        x_t = torch.from_numpy(x_f32)[None, None].to(device)
        t_bone, _ = teacher_model(x_t)
        b_np = t_bone[0, 0].clamp(0.0, 1.0).cpu().numpy()
        cached_targets.append((b_np * 255.0).round().astype(np.uint8))

del teacher_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"[INFO] Extracted {len(cached_targets)} bone targets. Teacher model released from VRAM.")

## 5. Model Architecture (ResNet34-UNet)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.utils.data import Dataset, DataLoader

class DistillationDataset(Dataset):
    def __init__(self, raw_list, target_list):
        self.raw_list = raw_list
        self.target_list = target_list

    def __len__(self):
        return len(self.raw_list)
        
    def __getitem__(self, idx):
        x = torch.from_numpy(self.raw_list[idx].astype(np.float32) / 255.0)[None, ...]
        b = torch.from_numpy(self.target_list[idx].astype(np.float32) / 255.0)[None, ...]
        return x, b

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, skip=None):
        # Bilinear upsampling avoids checkerboard artifacts
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x

class ResNet34UNet(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        
        # Adapt first conv layer for 1-channel grayscale input
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.conv1.weight.copy_(backbone.conv1.weight.mean(dim=1, keepdim=True))

        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 64)
        self.dec0 = DecoderBlock(64, 0, 32)

        self.final_conv = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x_pool = self.maxpool(x0)
        x1 = self.layer1(x_pool)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)

        d4 = self.dec4(x4, x3)
        d3 = self.dec3(d4, x2)
        d2 = self.dec2(d3, x1)
        d1 = self.dec1(d2, x0)
        d0 = self.dec0(d1)

        bone = torch.clamp(self.final_conv(d0), 0.0, 1.0)
        soft = torch.clamp(x - bone, 0.0, 1.0)
        return bone, soft

student_model = ResNet34UNet().to(device)
params_count = sum(p.numel() for p in student_model.parameters())
print(f"[INFO] ResNet34-UNet initialized ({params_count:,} parameters)")

## 6. Training Pipeline

In [ ]:
import torch.optim as optim

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler('cuda')
    def get_autocast():
        return autocast('cuda')
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    def get_autocast():
        return autocast()

sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], dtype=torch.float32, device=device).view(1, 1, 3, 3)
sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], dtype=torch.float32, device=device).view(1, 1, 3, 3)

def compute_edge_loss(pred, target):
    p_gx = F.conv2d(pred, sobel_x, padding=1)
    p_gy = F.conv2d(pred, sobel_y, padding=1)
    t_gx = F.conv2d(target, sobel_x, padding=1)
    t_gy = F.conv2d(target, sobel_y, padding=1)
    return F.l1_loss(p_gx, t_gx) + F.l1_loss(p_gy, t_gy)

dataset = DistillationDataset(raw_images, cached_targets)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True, pin_memory=True)
optimizer = optim.AdamW(student_model.parameters(), lr=3e-4, weight_decay=1e-5)

EPOCHS = 60
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f"[INFO] Starting training ({EPOCHS} epochs, {len(dataset)} samples)...")

for epoch in range(1, EPOCHS + 1):
    student_model.train()
    running_loss = 0.0

    for x_b, t_bone in train_loader:
        x_b = x_b.to(device)
        t_bone = t_bone.to(device)
        t_soft = torch.clamp(x_b - t_bone, 0.0, 1.0)

        optimizer.zero_grad()
        with get_autocast():
            s_bone, s_soft = student_model(x_b)

            # Bone-weighted L1 loss: emphasizes sparse bone structures to eliminate rib shadows
            weight_map = 1.0 + 5.0 * t_bone
            loss_bone = torch.mean(weight_map * torch.abs(s_bone - t_bone))

            loss_soft = F.l1_loss(s_soft, t_soft)
            loss_edge = compute_edge_loss(s_bone, t_bone)

            loss = loss_bone + 0.8 * loss_soft + 0.5 * loss_edge

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] - Loss: {avg_loss:.5f} (lr: {scheduler.get_last_lr()[0]:.6f})")
        torch.save(student_model.state_dict(), "checkpoint_latest.pth")

print("[INFO] Training complete.")

## 7. Model Evaluation

In [ ]:
import matplotlib.pyplot as plt

student_model.eval()
test_raw = raw_images[0]
test_x = torch.from_numpy(test_raw.astype(np.float32) / 255.0)[None, None].to(device)

with torch.no_grad():
    s_bone, s_soft = student_model(test_x)

s_soft_img = s_soft[0, 0].cpu().numpy()
s_bone_img = s_bone[0, 0].cpu().numpy()
target_bone_img = cached_targets[0].astype(np.float32) / 255.0

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].set_title("Input CXR")
axes[0].imshow(test_raw, cmap='gray')
axes[0].axis('off')

axes[1].set_title("Teacher Target (Soft-Tissue)")
axes[1].imshow(np.clip(test_raw.astype(float)/255.0 - target_bone_img, 0, 1), cmap='gray')
axes[1].axis('off')

axes[2].set_title("Student Output (Soft-Tissue)")
axes[2].imshow(s_soft_img, cmap='gray')
axes[2].axis('off')

axes[3].set_title("Suppressed Bone Mask")
axes[3].imshow(s_bone_img, cmap='inferno')
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 8. Export TorchScript Model

In [ ]:
student_model.eval()
dummy_input = torch.randn(1, 1, 1024, 1024).to(device)

traced_model = torch.jit.trace(student_model, dummy_input)
output_filename = "bone_suppression_commercial.ts"
traced_model.save(output_filename)

file_size_mb = os.path.getsize(output_filename) / (1024 * 1024)
print(f"[INFO] Exported {output_filename} ({file_size_mb:.2f} MB)")

## 9. Download Model Artifact

In [ ]:
from google.colab import files

files.download(output_filename)